In [1]:
"""
=============================================================
DSA COMPFEST 18 — Sigma3 V2
=============================================================
Base    : sigma3 EXACT (LB 12.00364) — ZERO fitur tambahan
Improve : + CatBoost sebagai model ke-3 dalam blend
          + Optuna 50 trials (dari 12)
          + Grid search 3-way blend weight
Target  : LB < 12.00364
=============================================================
"""

# ============================================================
# 01. IMPORT
# ============================================================
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ============================================================
# 02. KONFIGURASI
# ============================================================
SEED       = 42
N_FOLDS    = 5
TUNE_FOLDS = 3
N_TRIALS   = 50     # lebih banyak dari sigma3 asli (12)
TARGET     = 'property_organic_content'
ID_COL     = 'sample_id'

np.random.seed(SEED)

def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

# ============================================================
# 03. LOAD DATA
# ============================================================
DATA_DIR = '/kaggle/input/competitions/seleksi-data-science-academy-compfest-18'
OUT_DIR  = '/kaggle/working/'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')

train_ids = train[ID_COL].copy()
test_ids  = test[ID_COL].copy()

print(f"Train : {train.shape} | Test : {test.shape}")
print(f"Target skewness: {train[TARGET].skew():.4f}")

# ============================================================
# 04. FEATURE ENGINEERING
# ============================================================
# ⚠️  PELAJARAN DARI SEMUA SUBMISSION TIM:
# Setiap penambahan fitur → LB memburuk tanpa kecuali.
# sigma3 original (6 flags saja) = TERBAIK (12.00364)
# Sigma3 + 3 fitur kita → 12.46580 (LEBIH BURUK)
# Sigma2/Sigma4 (Noveline) → 12.39/12.40 (LEBIH BURUK)
# Kesimpulan: JANGAN tambah fitur apapun.

CAT_COLS = [
    'source_id', 'has_band_A_spectrum', 'has_band_B_spectrum',
    'sampling_strategy', 'sampling_depth_cm', 'geo_zone_macro',
    'geo_zone_micro', 'geo_zone_meso', 'land_cover_type',
    'biome', 'parent_rock_type'
]

def feature_engineer(df):
    df = df.copy()
    # 6 missing flags — persis sigma3 original, tidak lebih tidak kurang
    df['flag_has_coord']     = df['latitude'].notna().astype(int)
    df['flag_has_acidity']   = df['property_acidity_index'].notna().astype(int)
    df['flag_has_cation_na'] = df['cation_Na'].notna().astype(int)
    df['flag_has_cation_set']= df['cation_Ca'].notna().astype(int)
    df['flag_has_particle']  = df['property_particle_coarse'].notna().astype(int)
    df['flag_has_band_B']    = (df['has_band_B_spectrum'] == 'YES').astype(int)
    for c in CAT_COLS:
        df[c] = df[c].astype('category')
    return df

train_fe = feature_engineer(train)
test_fe  = feature_engineer(test)

# Align categories — sigma3 pattern
for c in CAT_COLS:
    cats = pd.api.types.union_categoricals(
        [train_fe[c], test_fe[c]]
    ).categories
    train_fe[c] = train_fe[c].cat.set_categories(cats)
    test_fe[c]  = test_fe[c].cat.set_categories(cats)

FEATURE_COLS = [c for c in train_fe.columns if c not in [ID_COL, TARGET]]
X      = train_fe[FEATURE_COLS]
y_raw  = train_fe[TARGET]
y      = np.log1p(y_raw)
X_test = test_fe[FEATURE_COLS]

print(f"Total fitur: {X.shape[1]}")  # harus 56 — sama dengan sigma3 asli

# XGBoost: integer codes
train_xgb = train_fe.copy()
test_xgb  = test_fe.copy()
for c in CAT_COLS:
    train_xgb[c] = train_xgb[c].cat.codes.replace(-1, np.nan)
    test_xgb[c]  = test_xgb[c].cat.codes.replace(-1, np.nan)
X_xgb      = train_xgb[FEATURE_COLS]
X_test_xgb = test_xgb[FEATURE_COLS]

# CatBoost: string (handle categoricals natively)
train_cb = train_fe.copy()
test_cb  = test_fe.copy()
for c in CAT_COLS:
    train_cb[c] = train_cb[c].astype(str).fillna('Unknown')
    test_cb[c]  = test_cb[c].astype(str).fillna('Unknown')
X_cb      = train_cb[FEATURE_COLS]
X_test_cb = test_cb[FEATURE_COLS]

# ============================================================
# 05. CV SETUP
# ============================================================
kf      = KFold(n_splits=N_FOLDS,    shuffle=True, random_state=SEED)
tune_kf = KFold(n_splits=TUNE_FOLDS, shuffle=True, random_state=SEED)

# ============================================================
# 06. OPTUNA — LightGBM (native API)
# ============================================================
def lgb_cv_rmse(params):
    oof = np.zeros(len(X))
    for ti, vi in tune_kf.split(X):
        dtr  = lgb.Dataset(X.iloc[ti], label=y.iloc[ti],
                           categorical_feature=CAT_COLS)
        dval = lgb.Dataset(X.iloc[vi], label=y.iloc[vi],
                           categorical_feature=CAT_COLS, reference=dtr)
        m = lgb.train(
            params, dtr, num_boost_round=2000, valid_sets=[dval],
            callbacks=[lgb.early_stopping(100, verbose=False),
                       lgb.log_evaluation(0)]
        )
        oof[vi] = m.predict(X.iloc[vi], num_iteration=m.best_iteration)
    return rmse(y, oof)

def lgb_objective(trial):
    return lgb_cv_rmse({
        'objective'        : 'regression',
        'metric'           : 'rmse',
        'verbosity'        : -1,
        'seed'             : SEED,
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 15, 200),
        'min_data_in_leaf' : trial.suggest_int('min_data_in_leaf', 5, 80),
        'feature_fraction' : trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction' : trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq'     : 1,
        'lambda_l1'        : trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
        'lambda_l2'        : trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
    })

print("\nTuning LightGBM...")
lgb_study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
lgb_study.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=False)
print(f"  Best LGB CV RMSE : {lgb_study.best_value:.5f}")
print(f"  Best params      : {lgb_study.best_params}")

# ============================================================
# 07. OPTUNA — XGBoost
# ============================================================
def xgb_objective(trial):
    params = {
        'objective'            : 'reg:squarederror',
        'tree_method'          : 'hist',
        'enable_categorical'   : False,
        'random_state'         : SEED,
        'n_estimators'         : 1500,
        'early_stopping_rounds': 50,
        'n_jobs'               : -1,
        'verbosity'            : 0,
        'learning_rate'        : trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'max_depth'            : trial.suggest_int('max_depth', 3, 10),
        'min_child_weight'     : trial.suggest_int('min_child_weight', 1, 20),
        'subsample'            : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree'     : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha'            : trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda'           : trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }
    oof = np.zeros(len(X_xgb))
    for ti, vi in tune_kf.split(X_xgb):
        m = xgb.XGBRegressor(**params)
        m.fit(X_xgb.iloc[ti], y.iloc[ti],
              eval_set=[(X_xgb.iloc[vi], y.iloc[vi])], verbose=False)
        oof[vi] = m.predict(X_xgb.iloc[vi])
    return rmse(y, oof)

print("\nTuning XGBoost...")
xgb_study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=False)
print(f"  Best XGB CV RMSE : {xgb_study.best_value:.5f}")
print(f"  Best params      : {xgb_study.best_params}")

# ============================================================
# 08. OPTUNA — CatBoost
# ============================================================
def cb_objective(trial):
    params = {
        'iterations'         : 1500,
        'loss_function'      : 'RMSE',
        'eval_metric'        : 'RMSE',
        'random_seed'        : SEED,
        'verbose'            : False,
        'allow_writing_files': False,
        'learning_rate'      : trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'depth'              : trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg'        : trial.suggest_float('l2_leaf_reg', 1.0, 10.0, log=True),
        'subsample'          : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bylevel'  : trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'min_data_in_leaf'   : trial.suggest_int('min_data_in_leaf', 1, 50),
    }
    oof = np.zeros(len(X_cb))
    cat_idx = [X_cb.columns.get_loc(c) for c in CAT_COLS]
    for ti, vi in tune_kf.split(X_cb):
        m = cb.CatBoostRegressor(**params)
        m.fit(X_cb.iloc[ti], y.iloc[ti],
              cat_features=CAT_COLS,
              eval_set=(X_cb.iloc[vi], y.iloc[vi]),
              early_stopping_rounds=80, verbose=False)
        oof[vi] = m.predict(X_cb.iloc[vi])
    return rmse(y, oof)

print("\nTuning CatBoost...")
cb_study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
cb_study.optimize(cb_objective, n_trials=N_TRIALS, show_progress_bar=False)
print(f"  Best CB CV RMSE  : {cb_study.best_value:.5f}")
print(f"  Best params      : {cb_study.best_params}")

# ============================================================
# 09. FINAL TRAINING — 5-Fold
# ============================================================
best_lgb_params = {
    'objective'   : 'regression',
    'metric'      : 'rmse',
    'verbosity'   : -1,
    'seed'        : SEED,
    'bagging_freq': 1,
    **lgb_study.best_params,
}
best_xgb_params = {
    'objective'            : 'reg:squarederror',
    'tree_method'          : 'hist',
    'enable_categorical'   : False,
    'random_state'         : SEED,
    'n_estimators'         : 3000,
    'early_stopping_rounds': 100,
    'n_jobs'               : -1,
    'verbosity'            : 0,
    **xgb_study.best_params,
}
best_cb_params = {
    'iterations'         : 3000,
    'loss_function'      : 'RMSE',
    'eval_metric'        : 'RMSE',
    'random_seed'        : SEED,
    'verbose'            : False,
    'allow_writing_files': False,
    **cb_study.best_params,
}

oof_lgb = np.zeros(len(X));     test_lgb = np.zeros(len(X_test))
oof_xgb = np.zeros(len(X_xgb)); test_xgb_pred = np.zeros(len(X_test_xgb))
oof_cb  = np.zeros(len(X_cb));  test_cb_pred  = np.zeros(len(X_test_cb))

print("\nFinal training 5-fold (LGB + XGB + CatBoost)...")
for fold_num, (ti, vi) in enumerate(kf.split(X)):
    # LightGBM — native API
    dtr  = lgb.Dataset(X.iloc[ti], label=y.iloc[ti],
                       categorical_feature=CAT_COLS)
    dval = lgb.Dataset(X.iloc[vi], label=y.iloc[vi],
                       categorical_feature=CAT_COLS, reference=dtr)
    m_lgb = lgb.train(
        best_lgb_params, dtr, num_boost_round=3000, valid_sets=[dval],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)]
    )
    oof_lgb[vi]   = m_lgb.predict(X.iloc[vi], num_iteration=m_lgb.best_iteration)
    test_lgb     += m_lgb.predict(X_test,     num_iteration=m_lgb.best_iteration) / N_FOLDS

    # XGBoost
    m_xgb = xgb.XGBRegressor(**best_xgb_params)
    m_xgb.fit(X_xgb.iloc[ti], y.iloc[ti],
              eval_set=[(X_xgb.iloc[vi], y.iloc[vi])], verbose=False)
    oof_xgb[vi]    = m_xgb.predict(X_xgb.iloc[vi])
    test_xgb_pred += m_xgb.predict(X_test_xgb) / N_FOLDS

    # CatBoost
    m_cb = cb.CatBoostRegressor(**best_cb_params)
    m_cb.fit(X_cb.iloc[ti], y.iloc[ti],
             cat_features=CAT_COLS,
             eval_set=(X_cb.iloc[vi], y.iloc[vi]),
             early_stopping_rounds=100, verbose=False)
    oof_cb[vi]    = m_cb.predict(X_cb.iloc[vi])
    test_cb_pred += m_cb.predict(X_test_cb) / N_FOLDS

    fl = rmse(y.iloc[vi], oof_lgb[vi])
    fx = rmse(y.iloc[vi], oof_xgb[vi])
    fc = rmse(y.iloc[vi], oof_cb[vi])
    print(f"  Fold {fold_num+1}: LGB={fl:.5f} | XGB={fx:.5f} | CB={fc:.5f}")

print(f"\nOOF log RMSE:")
print(f"  LGB: {rmse(y, oof_lgb):.5f}")
print(f"  XGB: {rmse(y, oof_xgb):.5f}")
print(f"  CB : {rmse(y, oof_cb):.5f}")

# ============================================================
# 10. GRID SEARCH 3-WAY BLEND
# ============================================================
print("\nGrid search blend weights (3-way)...")
best_w, best_blend_rmse = (0.75, 0.25, 0.0), 1e9

for wl in np.arange(0.0, 1.01, 0.05):
    for wx in np.arange(0.0, 1.01 - wl, 0.05):
        wc = round(1.0 - wl - wx, 10)
        if wc < 0:
            continue
        blend = wl * oof_lgb + wx * oof_xgb + wc * oof_cb
        r = rmse(y, blend)
        if r < best_blend_rmse:
            best_blend_rmse = r
            best_w = (wl, wx, wc)

wl, wx, wc = best_w
print(f"Best blend: LGB={wl:.2f} | XGB={wx:.2f} | CB={wc:.2f}")
print(f"OOF log RMSE  : {best_blend_rmse:.5f}")

oof_blend = wl * oof_lgb + wx * oof_xgb + wc * oof_cb
print(f"OOF RMSE asli : {rmse(np.expm1(y), np.expm1(oof_blend)):.4f}")

# Bandingkan dengan sigma3 original (LGB 75% + XGB 25%)
sigma3_oof = 0.75 * oof_lgb + 0.25 * oof_xgb
print(f"Sigma3 blend  : {rmse(y, sigma3_oof):.5f}  (benchmark)")

# ============================================================
# 11. GENERATE SUBMISSION
# ============================================================
test_blend = wl * test_lgb + wx * test_xgb_pred + wc * test_cb_pred
test_blend  = np.clip(np.expm1(test_blend), a_min=0, a_max=None)

submission = pd.DataFrame({ID_COL: test_ids, TARGET: test_blend})

assert len(submission) == 2670
assert submission[TARGET].isna().sum() == 0
assert (submission[TARGET] < 0).sum() == 0

submission.to_csv('submission.csv', index=False)
print(f"\nsubmission.csv saved — {submission.shape}")
print(submission.describe())

# OOF untuk Research Questions
pd.DataFrame({
    'sample_id': train_ids,
    'y_true'   : y_raw.values,
    'oof_lgb'  : np.expm1(oof_lgb),
    'oof_xgb'  : np.expm1(oof_xgb),
    'oof_cb'   : np.expm1(oof_cb),
    'oof_blend': np.expm1(oof_blend),
}).to_csv('oof_predictions.csv', index=False)
print("oof_predictions.csv saved")

Train : (11210, 52) | Test : (2670, 51)
Target skewness: 2.0278
Total fitur: 56

Tuning LightGBM...
  Best LGB CV RMSE : 0.28979
  Best params      : {'learning_rate': 0.012772278261764195, 'num_leaves': 167, 'min_data_in_leaf': 43, 'feature_fraction': 0.6476084733072378, 'bagging_fraction': 0.6046989185898562, 'lambda_l1': 0.005874957066385822, 'lambda_l2': 0.0010268874874202268}

Tuning XGBoost...
  Best XGB CV RMSE : 0.29602
  Best params      : {'learning_rate': 0.01061538832924292, 'max_depth': 8, 'min_child_weight': 19, 'subsample': 0.6343057804229745, 'colsample_bytree': 0.6309462732899698, 'reg_alpha': 0.02175975253721987, 'reg_lambda': 0.0010734360502063001}

Tuning CatBoost...
  Best CB CV RMSE  : 0.29554
  Best params      : {'learning_rate': 0.032383173738495544, 'depth': 9, 'l2_leaf_reg': 8.274534967431347, 'subsample': 0.6197828939074657, 'colsample_bylevel': 0.6110571607286253, 'min_data_in_leaf': 12}

Final training 5-fold (LGB + XGB + CatBoost)...
  Fold 1: LGB=0.28659